# Job-Skill Bridge Table Loader

This notebook maintains the `warehouse.bridge_job_skill` bridge table.

**Purpose**: Connect jobs to skills with evidence-based relationships

**Key Features**:
* Many-to-many relationship between jobs and skills
* Confidence scores and evidence types
* Temporal alignment with job SCD2 versions (effective_from/effective_to)
* Bridge records follow job version lifecycle

**Sources**: 
* `semantic.sem_job_skill_evidence` (skill evidence)
* `warehouse.dim_job` (job versions)
* `warehouse.dim_skill` (skills)

**Target**: `workspace.warehouse.bridge_job_skill`

In [0]:
%sql
-- Extract job-skill evidence from semantic layer
CREATE OR REPLACE TEMP VIEW skill_evidence_extract AS
SELECT 
  enterprise_job_id,
  skill_id,
  skill_name,
  confidence_score,
  evidence_type,
  source_priority,
  extracted_at
FROM workspace.semantic.sem_job_skill_evidence
WHERE enterprise_job_id IS NOT NULL
  AND skill_id IS NOT NULL;

In [0]:
%sql
-- Resolve to warehouse job_sk and skill_sk
-- Note: Only link to current job versions to avoid duplicating skills across versions
CREATE OR REPLACE TEMP VIEW bridge_with_keys AS
SELECT 
  e.enterprise_job_id,
  j.job_sk,
  s.skill_sk,
  e.confidence_score as confidence,
  e.evidence_type,
  e.source_priority,
  j.effective_from,
  j.effective_to,
  j.is_current
FROM skill_evidence_extract e
INNER JOIN workspace.warehouse.dim_job j
  ON e.enterprise_job_id = j.enterprise_job_id
  AND j.is_current = TRUE  -- Only current job versions
INNER JOIN workspace.warehouse.dim_skill s
  ON LOWER(TRIM(e.skill_name)) = LOWER(TRIM(s.skill_name))  -- Join on skill_name since skill_ids don't match
WHERE j.job_sk IS NOT NULL
  AND s.skill_sk IS NOT NULL;

In [0]:
%sql
-- Deduplicate bridge_with_keys by (job_sk, skill_sk)
CREATE OR REPLACE TEMP VIEW bridge_deduped AS
SELECT *,
  ROW_NUMBER() OVER (
    PARTITION BY job_sk, skill_sk
    ORDER BY confidence DESC, source_priority ASC, effective_from DESC
  ) as rn
FROM bridge_with_keys;

-- Generate bridge surrogate keys
CREATE OR REPLACE TEMP VIEW bridge_final AS
SELECT
  COALESCE(b.bridge_job_skill_sk, 
    ROW_NUMBER() OVER (ORDER BY bk.job_sk, bk.skill_sk) + COALESCE(max_sk, 0)) as bridge_job_skill_sk,
  bk.job_sk,
  bk.skill_sk,
  bk.confidence,
  bk.evidence_type,
  bk.source_priority,
  bk.effective_from,
  bk.effective_to,
  bk.is_current
FROM (
  SELECT 
    job_sk,
    skill_sk,
    confidence,
    evidence_type,
    source_priority,
    effective_from,
    effective_to,
    is_current
  FROM bridge_deduped
  WHERE rn = 1
) bk
LEFT JOIN (
  SELECT job_sk, skill_sk, MAX(bridge_job_skill_sk) as bridge_job_skill_sk
  FROM workspace.warehouse.bridge_job_skill
  WHERE is_current = TRUE
  GROUP BY job_sk, skill_sk
) b
  ON bk.job_sk = b.job_sk
  AND bk.skill_sk = b.skill_sk
CROSS JOIN (
  SELECT COALESCE(MAX(bridge_job_skill_sk), 0) as max_sk 
  FROM workspace.warehouse.bridge_job_skill
) max_keys;

In [0]:
%sql
-- Close existing bridge records when job is updated
-- This happens when a job gets a new version in dim_job
MERGE INTO workspace.warehouse.bridge_job_skill target
USING (
  SELECT DISTINCT
    b.bridge_job_skill_sk,
    j.effective_to
  FROM workspace.warehouse.bridge_job_skill b
  INNER JOIN workspace.warehouse.dim_job j
    ON b.job_sk = j.job_sk
  WHERE j.is_current = FALSE
    AND b.is_current = TRUE
    AND b.effective_to IS NULL
) source
ON target.bridge_job_skill_sk = source.bridge_job_skill_sk
WHEN MATCHED THEN UPDATE SET
  target.effective_to = source.effective_to,
  target.is_current = FALSE;

In [0]:
%sql
-- Merge into bridge table
MERGE INTO workspace.warehouse.bridge_job_skill target
USING bridge_final source
ON target.job_sk = source.job_sk
  AND target.skill_sk = source.skill_sk
  AND target.is_current = TRUE
WHEN MATCHED THEN UPDATE SET
  target.confidence = source.confidence,
  target.evidence_type = source.evidence_type,
  target.source_priority = source.source_priority,
  target.effective_from = source.effective_from
WHEN NOT MATCHED THEN INSERT (
  bridge_job_skill_sk,
  job_sk,
  skill_sk,
  confidence,
  evidence_type,
  source_priority,
  effective_from,
  effective_to,
  is_current
) VALUES (
  source.bridge_job_skill_sk,
  source.job_sk,
  source.skill_sk,
  source.confidence,
  source.evidence_type,
  source.source_priority,
  source.effective_from,
  source.effective_to,
  source.is_current
);

In [0]:
%sql
-- Validate bridge table
SELECT 
  COUNT(*) as total_bridge_records,
  COUNT(DISTINCT job_sk) as jobs_with_skills,
  COUNT(DISTINCT skill_sk) as skills_mapped,
  AVG(confidence) as avg_confidence,
  SUM(CASE WHEN is_current THEN 1 ELSE 0 END) as current_mappings
FROM workspace.warehouse.bridge_job_skill;

-- Sample bridge records with job and skill details
SELECT 
  b.bridge_job_skill_sk,
  b.job_sk,
  j.enterprise_job_id,
  j.title_normalized,
  b.skill_sk,
  s.skill_name,
  b.confidence,
  b.evidence_type,
  b.is_current
FROM workspace.warehouse.bridge_job_skill b
INNER JOIN workspace.warehouse.dim_job j ON b.job_sk = j.job_sk
INNER JOIN workspace.warehouse.dim_skill s ON b.skill_sk = s.skill_sk
WHERE b.is_current = TRUE
ORDER BY b.job_sk, b.confidence DESC
LIMIT 20;